# zero-grad-set-none — worked example 3: zero_grad must follow step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `zero-grad-set-none`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The correct training-step order is `backward → step → zero_grad`. If `zero_grad` runs between `backward` and `step`, the gradients are wiped before the optimizer reads them, so parameters never update and the loss stays flat. The fix is to clear grads only AFTER stepping.

## Worked solution

We run a correctly ordered training step on a quadratic.

1. Forward and loss: `pred = model(x)`, then a squared error.
2. `loss.backward()` populates the gradients.
3. `opt.step()` updates the parameters USING those gradients.
4. Only THEN do we set each `p.grad = None` to clear for the next iteration.
5. We return the pre-update loss value.

We run several steps and print that the loss decreases monotonically — the signature of a correctly ordered loop.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)
model = nn.Linear(3, 1)
opt = t.optim.SGD(model.parameters(), lr=0.05)
x = t.randn(16, 3)
y = t.randn(16, 1)
loss_fn = nn.MSELoss()

def fixed_step(model, opt, x, y, loss_fn):
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    for p in model.parameters():
        p.grad = None
    return loss.item()

losses = [fixed_step(model, opt, x, y, loss_fn) for _ in range(5)]
print('losses:', [round(l, 4) for l in losses])
print('decreasing:', all(losses[i] >= losses[i + 1] for i in range(len(losses) - 1)))